# 02 — Fold assignment and leakage check

This notebook confirms the per-fold train/val/test assignment produced by `FoldPlanner.plan`. For each fold the test block is the fold index, the validation block is the next block (cyclically), and the remaining blocks form the training set. We verify the three splits are mutually disjoint, so no azimuth sample leaks between them.

Modules exercised: `pipelines.cross_validation_pipeline.folds.FoldPlanner`, `tools.regions.SplitRegions`, `tools.regions.CropRegion`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Build the planner and enumerate all fold plans

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

config                     = CrossValidationConfig()
config.folds.n_folds       = 8
config.folds.azimuth_start = 0
config.folds.azimuth_end   = 800

planner = FoldPlanner(config, range_start=0, range_end=100)
plans   = planner.plans()

for plan in plans:
    print(f"fold {plan.fold_index}: test_block={plan.test_block}  val_block={plan.val_block}  train_blocks={plan.train_blocks}")

## Confirm disjointness from the block-index assignment

Leakage at the block level would show up as a block appearing in more than one split. We assert that the test block, validation block, and training blocks form a disjoint cover of all block indices.

In [ ]:
all_blocks = set(range(config.folds.n_folds))

for plan in plans:
    test  = {plan.test_block}
    val   = {plan.val_block}
    train = set(plan.train_blocks)

    disjoint = not (test & val) and not (test & train) and not (val & train)
    cover    = (test | val | train) == all_blocks

    print(f"fold {plan.fold_index}: disjoint={disjoint}  full_cover={cover}")

## Render the assignment as a fold-by-block matrix

Rows are folds, columns are azimuth blocks. Each cell is coloured by its role (train, validation, test). A leak would appear as two roles claiming the same cell, which is impossible to draw here because each cell holds exactly one role.

In [ ]:
import matplotlib.colors as mcolors

ROLE_TRAIN, ROLE_VAL, ROLE_TEST = 0, 1, 2
matrix = np.full((len(plans), config.folds.n_folds), ROLE_TRAIN, dtype=int)

for plan in plans:
    matrix[plan.fold_index, plan.val_block]  = ROLE_VAL
    matrix[plan.fold_index, plan.test_block] = ROLE_TEST

cmap   = mcolors.ListedColormap(["#cfe8cf", "#f4c879", "#d96c6c"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0, vmax=2)

ax.set_xticks(range(config.folds.n_folds))
ax.set_yticks(range(len(plans)))
ax.set_xlabel("azimuth block index")
ax.set_ylabel("fold index")
ax.set_title("Per-fold block assignment (green train, amber val, red test)")

for plan in plans:
    for block in range(config.folds.n_folds):
        label = {ROLE_TRAIN: "tr", ROLE_VAL: "val", ROLE_TEST: "te"}[matrix[plan.fold_index, block]]
        ax.text(block, plan.fold_index, label, ha="center", va="center", fontsize=8)

plt.show()

## Expected visual outcome

Every fold reports `disjoint = True` and `full_cover = True`. The matrix shows exactly one red (test) cell and one amber (validation) cell per row, with the test cell on the diagonal and the validation cell one step to its right (wrapping at the last fold). All remaining cells are green training blocks.